# Load and Split Dataset
Split dataset into:
- Train --> 80%
- Validation --> 10%
- Test --> 10%

In [2]:
from datasets import load_dataset, DatasetDict

dataset = load_dataset(
    "json",
    data_files="../data/raw/english_pii_43k.jsonl",
    split="train"
)
# split dataset into train and test splits
first_split = dataset.train_test_split(test_size=0.2, seed=42)
# split dataset into test and validation splits
second_split = first_split["test"].train_test_split(
    test_size=0.5,
    seed=42
)

splits = DatasetDict({
    "train": first_split["train"],
    "validation": second_split["train"],
    "test": second_split["test"]
})

print({name: len(data) for name, data in splits.items()})


{'train': 34800, 'validation': 4350, 'test': 4351}


# Label Distribution and coverage
ensure that the 13 targted labels are present in all splits

In [3]:
from collections import Counter

TARGET_LABELS = {
    "PASSWORD", "USERNAME", "EMAIL", "PHONENUMBER",
    "FIRSTNAME", "LASTNAME", "SSN",
    "CREDITCARDNUMBER", "CREDITCARDCVV",
    "IBAN", "ACCOUNTNUMBER", "STREET", "IPV4"
}

label_counts = {} # to save the result for each data split

for name, data in splits.items():
    counts = Counter( # counter returns a dict, here it returns labels as keys and values as number of label (entities) presented in each data split
        annotation["label"]
        for record in data
        for annotation in record["privacy_mask"]
        if annotation["label"] in TARGET_LABELS
    )
    label_counts[name] = counts

print(f"{'Label':<20} {'Train':>8} {'Val':>8} {'Test':>8}") # formated text

for label in sorted(TARGET_LABELS): # we sorted the labels first in alphetic order
    print(
        f"{label:<20} "
        f"{label_counts['train'][label]:>8} "
        f"{label_counts['validation'][label]:>8} "
        f"{label_counts['test'][label]:>8}"
    )


Label                   Train      Val     Test
ACCOUNTNUMBER            2262      301      268
CREDITCARDCVV             690       93       86
CREDITCARDNUMBER         2034      268      272
EMAIL                    3234      441      417
FIRSTNAME               10645     1319     1359
IBAN                     1601      187      185
IPV4                     2224      256      240
LASTNAME                 3836      498      483
PASSWORD                 2098      246      278
PHONENUMBER              1924      262      239
SSN                      1630      207      208
STREET                   2249      264      286
USERNAME                 2433      325      298


# Data Leakage
ensure that there is no identical texts or templates ditributed across splits

In [4]:
from itertools import combinations # used to incpect intersection between the three data splits
# function to convert text samples into template, so we can replace annotated entities with their labels
def get_template(record):
    text = record["source_text"]

    annotations = sorted(
        record["privacy_mask"],
        key=lambda x: x["start"],
        reverse=True # sorting annotations in a desending order. why? لbeacuse replacement may change the offset 
        #so if we start replacing the entities on the right of the text first the remaining text won't be affected
    )

    for annotation in annotations:
        start = annotation["start"]
        end = annotation["end"]
        label = annotation["label"]

        text = text[:start] + f"<{label}>" + text[end:]

    return " ".join(text.lower().split())


templates = {
    name: {get_template(record) for record in data}
    for name, data in splits.items()
}

for group_a, group_b in combinations(templates, 2):
    overlap = templates[group_a] & templates[group_b]
    print(f"{group_a} vs {group_b}: {len(overlap)} shared templates")


train vs validation: 1 shared templates
train vs test: 1 shared templates
validation vs test: 1 shared templates


- Calculating shared templates and the number of their samples

In [5]:
template_counts = {
    name: Counter(get_template(record) for record in data) # count number of templates in each data split 
    for name, data in splits.items()
}

shared_templates = set()

for a, b in combinations(template_counts, 2):
    shared_templates.update(
        template_counts[a].keys() & template_counts[b].keys() # find intersection
    )

print("Distinct shared templates:", len(shared_templates))

for i, template in enumerate(shared_templates, 1):
    print(f"\nTemplate {i}")
    for name in splits:
        print(f"{name}: {template_counts[name][template]} records")


Distinct shared templates: 3

Template 1
train: 1 records
validation: 1 records
test: 0 records

Template 2
train: 0 records
validation: 1 records
test: 1 records

Template 3
train: 1 records
validation: 0 records
test: 1 records


- Handling data leakage by moveing samples between splits

In [6]:
from datasets import concatenate_datasets
# function recives source and destination datasets and moves samples between them
def move_shared(source, destination):
    destination_templates = {
        get_template(record) for record in splits[destination]
    }
# use indices to save template index that intersect between source and desination
    move_indices = [
        i for i, record in enumerate(splits[source])
        if get_template(record) in destination_templates
    ]

    move_set = set(move_indices)

    moving = splits[source].select(move_indices)
# removing moving samples from records
    splits[source] = splits[source].select(
        [i for i in range(len(splits[source])) if i not in move_set]
    )
# moving samples to destination
    splits[destination] = concatenate_datasets(
        [splits[destination], moving]
    )

# execution order is important since each call change splits
move_shared("validation", "train")
move_shared("test", "train")
move_shared("test", "validation")


- Ensure data spliting integrity after handling data leakage 

- Ensure removing shared templates

In [7]:
templates = {
    name: {get_template(record) for record in data}
    for name, data in splits.items()
}

for a, b in combinations(templates, 2):
    overlap = templates[a] & templates[b]
    print(f"{a} vs {b}: {len(overlap)} shared templates")

print("\nFinal split sizes:")
for name, data in splits.items():
    print(f"{name}: {len(data)}")

print("Total:", sum(len(data) for data in splits.values()))


train vs validation: 0 shared templates
train vs test: 0 shared templates
validation vs test: 0 shared templates

Final split sizes:
train: 34802
validation: 4350
test: 4349
Total: 43501


- Check intersection after normalization

In [ ]:
import re


def normalize_template(record):
    template = get_template(record) # convert text to lower case and normalized white spaces
    return re.sub(r"[\W_]+", " ", template).strip() # The expression replaces sequences of punctuation marks, whitespace, 
# and underscores with a single space, while typically preserving letters and numbers.

normalized = {
    name: {normalize_template(record) for record in data}
    for name, data in splits.items()
}

for a, b in combinations(normalized, 2):
    overlap = normalized[a] & normalized[b]
    print(f"{a} vs {b}: {len(overlap)} normalized overlaps")

print("\nMissing target labels:")

for name, data in splits.items():
    counts = Counter(
        ann["label"]
        for record in data
        for ann in record["privacy_mask"]
        if ann["label"] in TARGET_LABELS
    )

    missing = TARGET_LABELS - counts.keys() # Set Difference to ensure target label distrbution between data splits
    print(f"{name}: {sorted(missing)}")


train vs validation: 0 normalized overlaps
train vs test: 0 normalized overlaps
validation vs test: 0 normalized overlaps

Missing target labels:
train: []
validation: []
test: []


# Save data splits after preprocessing

In [9]:
from pathlib import Path

save_path = Path("../data/processed/mbert_splits")

save_path.parent.mkdir(parents=True, exist_ok=True)

splits.save_to_disk(str(save_path))

print("Dataset splits saved successfully!")


Saving the dataset (1/1 shards): 100%|██████████| 4349/4349 [00:00<00:00, 53759.34 examples/s]

Dataset splits saved successfully!
